# Raw Data Preprocessor
This notebook cleans and prepares the raw data straight from the sensor to the desired format.

In [34]:
import os
import re
import pandas as pd
import shutil

In [35]:
#Declarations
DATE = '2026-07-27'
META_FILE_PATH = f'../data/00_metadata/{DATE}.csv'
RAW_FOLDER_PATH = f'../data/01_raw/{DATE}'
PROC_FOLDER_PATH = f"../data/02_preprocessed_test/{DATE}"
SHIFT_INDEX = 0                # = [1 - {FILE SEQ}]
INIT_TRIAL_NO_SHIFT = 0         # = trial no. start - 1

In [36]:
if os.path.exists(PROC_FOLDER_PATH):                           # Delete the existing folder and all its contents
    shutil.rmtree(PROC_FOLDER_PATH)
os.makedirs(PROC_FOLDER_PATH)                                  # Create a new empty folder
files = os.listdir(RAW_FOLDER_PATH)                            # Create a list containing the file names of the raw files

csv_files = [
    file
    for file in files
    if ".csv" in file.lower()
    #if file.lower().endswith(".csv_files")
]

csv_files = sorted(
    csv_files,
    key=lambda file: int(
        re.search(r"\((\d+)\)", file).group(1)
    )
)

In [37]:
for c in csv_files:
    match = re.search(r"\((\d+)\)", c)
    #print(match)

In [38]:
# Get run number to match with experiment logs
run_index = {}

for c in csv_files:
    match = re.search(r"\((\d+)\)", c)
    run_no = int(match.group(1)) if match else None
    run_no = run_no + SHIFT_INDEX
    run_index[run_no] = c

run_index = dict(sorted(run_index.items()))

In [39]:
metadata = pd.read_csv(META_FILE_PATH)
metadata['File'] = metadata['Run_no'].apply(lambda x: run_index[x])
#metadata.head(51)

In [40]:
metadata

,DATE,Material,k,Mass,Volume,rho,cp,Trial,RH_percent,Material_T0_C,Sensor_T0_C,Run_no,Timestamp,Valid,File
0,2026/07/27,PS,NaN,NaN,NaN,NaN,NaN,1,49,31.3,25.4,1,234541,True,I_V-t Sampling [(1) ; 2026_07_26 23_45_41].csv
1,2026/07/27,PS,NaN,NaN,NaN,NaN,NaN,2,49,30.9,25.6,2,234644,True,I_V-t Sampling [(2) ; 2026_07_26 23_46_44].csv
2,2026/07/27,PS,NaN,NaN,NaN,NaN,NaN,3,49,30.8,25.3,3,234741,True,I_V-t Sampling [(3) ; 2026_07_26 23_47_41].csv
3,2026/07/27,PS,NaN,NaN,NaN,NaN,NaN,4,49,30.8,25.3,4,234921,True,I_V-t Sampling [(4) ; 2026_07_26 23_49_21].csv
4,2026/07/27,PS,NaN,NaN,NaN,NaN,NaN,5,48,30.7,25.4,5,235129,True,I_V-t Sampling [(5) ; 2026_07_26 23_51_29].csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,2026/07/27,Cu,NaN,NaN,NaN,NaN,NaN,2,48,30.4,25.3,80,21524,True,I_V-t Sampling [(80) ; 2026_07_27 2_15_24].csv
80,2026/07/27,Cu,NaN,NaN,NaN,NaN,NaN,3,48,30.4,25.6,81,21619,True,I_V-t Sampling [(81) ; 2026_07_27 2_16_19].csv
81,2026/07/27,Cu,NaN,NaN,NaN,NaN,NaN,4,48,30.3,25.2,82,21715,True,I_V-t Sampling [(82) ; 2026_07_27 2_17_15].csv
82,2026/07/27,Cu,NaN,NaN,NaN,NaN,NaN,5,47,30.4,25.1,83,21812,True,I_V-t Sampling [(83) ; 2026_07_27 2_18_12].csv


In [41]:
max_trial = int(metadata["Trial"].max())
zero_pad = len(str(max_trial))


for index, row in metadata.iterrows():

    valid = row["Valid"]

    if valid:
        file = row["File"]

        material = (
            str(row["Material"])
            .strip()
            .lower()
            .replace(" ", "_")
        )

        trial_no = int(row["Trial"])
        therm_cond = float(row["k"])
        mass = float(row["Mass"])
        vol = float(row["Volume"])
        density = float(row["rho"])
        heatcap = float(row["cp"])

        file_no = trial_no + INIT_TRIAL_NO_SHIFT

        file_name = (
            f"{PROC_FOLDER_PATH}/{material}_"
            #f"{trial_no:0{zero_pad}d}.csv"
            f"{file_no:0{zero_pad}d}.csv"
        )

        df = pd.read_csv(
            f"{RAW_FOLDER_PATH}/{file}",
            skiprows=255
        )

        # Clean column names
        df.columns = df.columns.str.strip()

        # Keep and rename the required sensor columns
        df = (
            df[["Time", "I1", "I2"]]
            .rename(columns={
                "I1": "Primary",
                "I2": "Secondary"
            })
            .copy()
        )

        # Add experiment information
        df["Sample"] = material
        df["Trial"] = trial_no + INIT_TRIAL_NO_SHIFT
        df["k"] = therm_cond
        df["Mass"] = mass
        df["Volume"] = vol
        df["rho"] = density
        df["cp"] = heatcap

        # Clean and organize rows
        df = (
            df
            .dropna(subset=["Time", "Primary", "Secondary"])
            .sort_values("Time")
            .reset_index(drop=True)
        )

        # Create a new sequential index column
        df.insert(0, "index", range(len(df)))

        # Final column order
        df = df[
            [
                "index",
                "Sample",
                "Trial",
                "k",
                "Mass",
                "Volume",
                "rho",
                "cp",
                "Time",
                "Primary",
                "Secondary"
            ]
        ]

        df.to_csv(file_name, index=False)

    else:
        print(row)